# 05 Assess Inversions
Visualizing the data was useful, but we need a more rigorous solution to evaluate the results. Specifically, we are interested in the false and true positive/negative rates for each of the treatments. I suppose as this document unfolds I'll narrow down exactly what that looks like, but for now lets start with loading *all* the variants into a tidy table. We'll borrow the formatting code from `read_data()` used in the visualization steps, but it needs to be augmented to do _everything_ and label it all.

In [26]:
library(dplyr)

In [43]:
read_truth_data <- function(size_treatment){
    truthfile <- paste0("simulated_data/inversions/", size_treatment, "/inv.", size_treatment,".vcf")

    true_inversions <- read.table(truthfile, header = F)[,c(1,2,8)]
    true_inversions$V8 <- as.numeric(unlist(lapply(true_inversions$V8, function(X){gsub(".+END=", "", X)})))
    names(true_inversions) <- c("contig", "position_start", "position_end")
    true_inversions$size <- size_treatment
    true_inversions <- group_by(true_inversions, contig) %>%
        mutate(id = 1:n())
    return(true_inversions[, c("contig", "id","size","position_start", "position_end")])
}

read_called_data <- function(size_treatment){
    called_df <- data.frame(
        sample = character(),
        contig = character(),
        size = character(),
        depth = character(),
        position_start = numeric(),
        position_end = numeric()
    )
    for(depth in c("05X", "2X", "5X", "10X", "20X")){
        samplesfile <- paste("simulated_data/called_sv/leviathan", size_treatment, depth , "by_sample/inversions.bedpe", sep = "/")
        poolfile <- paste("simulated_data/called_sv/leviathan", size_treatment, depth, "by_pop/inversions.bedpe", sep = "/")

        sample_inversions <- read.table(
            paste("simulated_data/called_sv/leviathan", size_treatment, depth, "by_sample/inversions.bedpe", sep = "/"),
            header = T
        )[,1:4]
        if( nrow(sample_inversions) > 0){
            sample_inversions$size <- size_treatment
            sample_inversions$depth <- depth
            sample_inversions$sample <- gsub(pattern = "\\..*X", "", sample_inversions$sample)
        } else {
            sample_inversions$size <- character()
            sample_inversions$depth <- character()
        }

        pooled_inversions <- read.table(poolfile, header = T)
        pooled_inversions <- pooled_inversions[,c("population","contig", "position_start", "position_end")]
        names(pooled_inversions)[1] <- "sample"
        if( nrow(pooled_inversions) > 0){
            pooled_inversions$sample <- "pooled"
            pooled_inversions$size <- size_treatment
            pooled_inversions$depth <- depth
        } else {
            pooled_inversions$size <- character()
            pooled_inversions$depth <- character()
        }
        called_df <- rbind(called_df, sample_inversions, pooled_inversions)
    }

    return(called_df)
}


Let's try this by having one dataframe of the known inversions and another of the called inversions. Using `Reduce(rbind, Map())` is going to make life a lot easier here.

In [41]:
truthset <- Reduce(rbind, Map(read_truth_data, c("small","medium","large","xl")))
truthset

contig,id,size,position_start,position_end
<chr>,<int>,<chr>,<int>,<dbl>
2L,1,small,3193196,3214221
2L,2,small,3940211,3944863
2L,3,small,13024404,13026577
2L,4,small,14845709,14861200
2L,5,small,20107901,20113695
2R,1,small,4328632,4333509
2R,2,small,9952840,9959287
2R,3,small,10569437,10578078
2R,4,small,11083686,11094907


In [44]:
calledset <- Reduce(rbind, Map(read_called_data, c("small","medium","large","xl")))
calledset

sample,contig,position_start,position_end,size,depth
<chr>,<chr>,<int>,<int>,<chr>,<chr>
pooled,2L,13024403,13026577,small,05X
sample_05,2L,14845709,14861199,small,2X
sample_05,2R,11083685,11094907,small,2X
sample_05,3R,20081036,20093594,small,2X
sample_05,3R,25379866,25402914,small,2X
sample_06,2L,3193195,3214221,small,2X
sample_04,2L,3940210,3944863,small,2X
sample_04,2L,14845708,14861199,small,2X
sample_04,3R,20081036,20093594,small,2X


The data is read in. So, now what? I think it might make sense to do a fuzzy-match for the inversions for a given size treatment. Fuzzy in the sense that the start/end positions of the called inversions can be something like 1kb away and from the known breakpoint and still count. Ideally, the output table should look something like:

| contig | size | start | end | TP | TN | FP | FN |
|:---|:---|:---|:---|:---|:---|:---|:---|
| 2R | small | 43324 | 43555 | 1 | 1 | 0 | 0 |

where `T` and `F` are True and False, respectively, and `P` and `N` are Positive and Negative

But, hold on, aren't we interested in the T/F N/P at all treatment levels? How would that look like?

| contig | size | depth |start | end | TP | TN | FP | FN |
|:---|:---|:---|:---|:---|:---|:---|:---|:---|
| 2R | small | 2X | 43324 | 43555 | 1 | 1 | 0 | 0 |

*And a final considertion:* not every sample has every simulated inversion. We need to get the table of which samples have which inversions and somehow cross-reference that to make sure that the presence/absence is actually true. How would that even look like? Do I have to add the presence/absence per inversion to the `truthset`? That would add a lot of columns, but maybe that's ok? Well, even if `sample_01` for `small` isn't the same as `sample_01` for `medium` (well, the SNPs would be the same but that's not relevant here), it _should_ suffice to have a column for `sample_01` through `sample_10` and one for `pool` (always true). That would give us one monolithic table of known inversions and whether a sample has it. It's worth a shot.

In [59]:
inv_inventory <- read.table("inversion_simulations.inventory", header = T)
names(inv_inventory)[4] <- "id"
head(inv_inventory)

,sample,size,contig,id,present,state
,<chr>,<chr>,<chr>,<int>,<lgl>,<chr>
1,sample_01,small,2R,1,TRUE,het
2,sample_01,small,2R,2,TRUE,het
3,sample_01,small,2R,3,TRUE,het
4,sample_01,small,2R,4,TRUE,het
5,sample_01,small,2R,5,TRUE,het
6,sample_02,small,2R,1,TRUE,het


We also need to add the pooled samples to this inventory

In [63]:
pool_inventory <- read.table("inversion_simulations.pool.inventory", header = T)
names(pool_inventory)[4] <- "id"
inv_inventory <- rbind(inv_inventory, pool_inventory)
tail(inv_inventory)

,sample,size,contig,id,present,state
,<chr>,<chr>,<chr>,<int>,<lgl>,<chr>
563,pool,large,3L,1,TRUE,hom
564,pool,large,3L,2,TRUE,hom
565,pool,xl,2R,1,TRUE,hom
566,pool,xl,2L,1,TRUE,hom
567,pool,xl,3R,1,TRUE,hom
568,pool,xl,3L,1,TRUE,hom


We need to somehow take this and combine it with `truthset` to look like:

| contig | id | size | position_start | position_end | sample_01 | sample_02 | etc. |
|:-------|:-------|:-------|:-------|:-------|:-------|:-------|:-------|
| 2R     | 1     |   small |    23 | 1032 | TRUE | FALSE | FALSE...|

Or maybe it would be better to have it in long format? Idk, let's try that first, it seems easier.

In [55]:
inv_inventory <- left_join(inv_inventory, truthset, by = c("contig","id", "size"))
head(inv_inventory)

,sample,size,contig,id,present,state,position_start,position_end
,<chr>,<chr>,<chr>,<int>,<lgl>,<chr>,<int>,<dbl>
1,sample_01,small,2R,1,TRUE,het,4328632,4333509
2,sample_01,small,2R,2,TRUE,het,9952840,9959287
3,sample_01,small,2R,3,TRUE,het,10569437,10578078
4,sample_01,small,2R,4,TRUE,het,11083686,11094907
5,sample_01,small,2R,5,TRUE,het,21736723,21759355
6,sample_02,small,2R,1,TRUE,het,4328632,4333509


Next comes what should be a big loop that goes through rows of `inv_inventory` (an inversion for a sample) and checks `calledset` for if that inversion was called. That would give us the overall, not taking depth into account. What if it makes sense to add `TP` and `FP` columns per depth? That would be 4x columns per depth treatment, which we can always elongate later. So the idea here is then to create all the columns (e.g. `TP_05X` is "True Positive 05X") as `FALSE` and adjust the value to `TRUE` if there is a match with the called variants

In [57]:
inv_inventory$TP_05X <- FALSE
inv_inventory$FP_05X <- FALSE
inv_inventory$TN_05X <- FALSE
inv_inventory$FN_05X <- FALSE

inv_inventory$TP_2X <- FALSE
inv_inventory$FP_2X <- FALSE
inv_inventory$TN_2X <- FALSE
inv_inventory$FN_2X <- FALSE

inv_inventory$TP_5X <- FALSE
inv_inventory$FP_5X <- FALSE
inv_inventory$TN_5X <- FALSE
inv_inventory$FN_5X <- FALSE

inv_inventory$TP_10X <- FALSE
inv_inventory$FP_10X <- FALSE
inv_inventory$TN_10X <- FALSE
inv_inventory$FN_10X <- FALSE

inv_inventory$TP_20X <- FALSE
inv_inventory$FP_20X <- FALSE
inv_inventory$TN_20X <- FALSE
inv_inventory$FN_20X <- FALSE

head(inv_inventory)

,sample,size,contig,id,present,state,position_start,position_end,TP_05X,FP_05X,...,TN_5X,FN_5X,TP_10X,FP_10X,TN_10X,FN_10X,TP_20X,FP_20X,TN_20X,FN_20X
,<chr>,<chr>,<chr>,<int>,<lgl>,<chr>,<int>,<dbl>,<lgl>,<lgl>,...,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>
1,sample_01,small,2R,1,TRUE,het,4328632,4333509,FALSE,FALSE,...,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
2,sample_01,small,2R,2,TRUE,het,9952840,9959287,FALSE,FALSE,...,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
3,sample_01,small,2R,3,TRUE,het,10569437,10578078,FALSE,FALSE,...,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
4,sample_01,small,2R,4,TRUE,het,11083686,11094907,FALSE,FALSE,...,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
5,sample_01,small,2R,5,TRUE,het,21736723,21759355,FALSE,FALSE,...,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE
6,sample_02,small,2R,1,TRUE,het,4328632,4333509,FALSE,FALSE,...,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE


So now what's left is to loop through `inv_inventory` and try to match calls in `calledset`, adjusting the column as necessary.